# - Fine-tuning dentaire avec Unsloth -
# Installation → Entraînement → Fusion LoRA → Export GGUF

# PARTIE 1 : INSTALLATION DES DÉPENDANCES

In [4]:
%pip install --upgrade \
    "torch>=2.4,<2.12" \
    "transformers==5.5.0" \
    "unsloth==2026.8.12" \
    "datasets==4.3.0" \
    "trl==0.24.0" \
    "peft==0.20.0" \
    "accelerate==1.14.0" \
    "bitsandbytes==0.50.0" \
    "protobuf==7.35.1"

print("✅ Installation terminée !")

Found existing installation: peft 0.20.0
Uninstalling peft-0.20.0:
  Successfully uninstalled peft-0.20.0
Found existing installation: unsloth 2026.8.11
Uninstalling unsloth-2026.8.11:
  Successfully uninstalled unsloth-2026.8.11
Found existing installation: bitsandbytes 0.50.0
Uninstalling bitsandbytes-0.50.0:
  Successfully uninstalled bitsandbytes-0.50.0
Found existing installation: trl 0.24.0
Uninstalling trl-0.24.0:
  Successfully uninstalled trl-0.24.0
Found existing installation: transformers 5.5.0
Uninstalling transformers-5.5.0:
  Successfully uninstalled transformers-5.5.0
Found existing installation: accelerate 1.14.0
Uninstalling accelerate-1.14.0:
  Successfully uninstalled accelerate-1.14.0
Found existing installation: protobuf 7.35.1
Uninstalling protobuf-7.35.1:
  Successfully uninstalled protobuf-7.35.1
Found existing installation: tokenizers 0.22.2
Uninstalling tokenizers-0.22.2:
  Successfully uninstalled tokenizers-0.22.2
  Using cached torchao-0.16.0-py3-none-any.w

# PARTIE 2 : ENTRAÎNEMENT (À exécuter APRÈS le redémarrage)

In [7]:
import json
import os
import sys
import dill
import torch
from datasets import load_dataset
from datasets.utils._dill import Pickler
from unsloth import FastLanguageModel
from trl import SFTTrainer, SFTConfig

# Compatibilité temporaire datasets 4.3 / Python 3.14
if sys.version_info >= (3, 14):
    def _batch_setitems_py314(self, items, obj=None):
        try:
            items = sorted(items)
        except Exception:
            from datasets.fingerprint import Hasher
            items = sorted(items, key=lambda item: Hasher.hash(item[0]))
        return dill.Pickler._batch_setitems(self, items, obj)

    Pickler._batch_setitems = _batch_setitems_py314

print("✅ Imports réussis")

✅ Imports réussis


## --- PARAMÈTRES ---

In [ ]:
max_seq_length = 2048
model_name = "unsloth/Qwen2.5-3B-bnb-4bit"
dataset_url = "/Users/elouan/Documents/Notebooks/dental_finetune_dataset_v2.json"
output_lora_dir = "lora_dental_model"
output_gguf_dir = "dental_model_gguf"
quantization_method = "q4_k_m"  # Options: q4_k_m, q8_0, f16

print(f"📦 Modèle : {model_name}")
print(f"🔧 Quantification GGUF : {quantization_method}")

## --- CHARGEMENT DU DATASET ---

In [ ]:
print("\n⏳ Chargement du dataset...")
dataset = load_dataset("json", data_files={"train": dataset_url}, split="train")

def format_example(example):
    response = example["response"]

    response_text = f"""
Diagnostic : {response.get('diagnostic', 'Non spécifié')}
Acte réalisé : {response.get('acte', 'Non spécifié')}
Anesthésie : {response.get('anesthesie', 'Non spécifié')}
Canaux traités : {', '.join(response.get('canaux', [])) if response.get('canaux') else 'Aucun'}
Irrigation : {response.get('irrigation', 'Aucune')}
Pansement : {response.get('pansement', 'Aucun')}
Code CCAM : {response.get('code_CCAM', 'Non spécifié')}
Prescription antibiotique : {response.get('prescription', {}).get('antibiotique', 'Aucun')}
Prescription antalgique : {response.get('prescription', {}).get('antalgique', 'Aucun')}
Prochain RDV : {response.get('prochain_rdv', 'Non spécifié')}"""

    example["text"] = f"### Instruction:\n{example['prompt']}\n\n### Response:{response_text}"
    return example

dataset = dataset.map(format_example)
print(f"✅ Dataset chargé : {len(dataset)} exemples")

## --- CHARGEMENT DU MODÈLE ---


In [ ]:
print(f"\n⏳ Chargement du modèle {model_name}...")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    load_in_4bit=True,
    dtype=None,  # Détection automatique : MLX sur Mac, CUDA sur Colab
)

print("✅ Modèle chargé avec succès !")

API HUGGING FACE ICI : https://huggingface.co/

## --- CONFIGURATION LORA ---

In [ ]:
print("\n⏳ Configuration LoRA...")

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
    max_seq_length=max_seq_length,
)

print("✅ LoRA configuré")

## --- ENTRAÎNEMENT ---


In [ ]:
print("\n⏳ Début de l'entraînement...")

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    tokenizer=tokenizer,
    args=SFTConfig(
        max_seq_length=max_seq_length,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        max_steps=200,              # Augmente à 500+ pour plus de précision
        logging_steps=1,
        output_dir="outputs_dental",
        optim="adamw_8bit",
        seed=3407,
        dataset_num_proc=1,
        report_to="none",
    ),
)

trainer.train()

print("✅ Entraînement terminé !")

## --- SAUVEGARDE LORA ---

In [ ]:
print("\n⏳ Sauvegarde du modèle LoRA...")
model.save_pretrained(output_lora_dir)
tokenizer.save_pretrained(output_lora_dir)
print(f"✅ Modèle LoRA sauvegardé dans '{output_lora_dir}'")

## --- TEST DU MODÈLE ENTRAÎNÉ ---

In [ ]:
print("\n🧪 Test du modèle entraîné :")

test_prompt = """### Instruction:
M. Dupont, 45 ans. Motif : douleur à la mastication. Dent 36. Test au froid : positif, percussion : négative.

### Response:"""

inputs = tokenizer(test_prompt, return_tensors="np")
outputs = model.generate(
    **inputs,
    max_new_tokens=512,
    temperature=0.7,
    do_sample=True,
)

result = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(result)

print("\n✅ Entraînement terminé avec succès !")

# PARTIE 3 : FUSION LORA + EXPORT GGUF


## Créer le dossier de sortie

In [ ]:
os.makedirs(output_gguf_dir, exist_ok=True)


## --- 1. Charger le modèle de base (16-bit) ---

In [ ]:
print("\n⏳ Chargement du modèle de base (16-bit)...")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    load_in_4bit=False,
    dtype=None,  # Détection automatique : MLX sur Mac, CUDA sur Colab
)

print("✅ Modèle de base chargé")

## --- 2. Charger les poids LoRA ---

In [ ]:
print("\n⏳ Chargement des poids LoRA...")
from peft import PeftModel

model = PeftModel.from_pretrained(model, output_lora_dir)
print("✅ Poids LoRA chargés")

## --- 3. Fusionner LoRA dans le modèle de base ---

In [ ]:
print("\n⏳ Fusion des poids LoRA...")
model = model.merge_and_unload()
print("✅ Fusion terminée")

## --- 4. Sauvegarder le modèle fusionné en HF ---

In [ ]:
print("\n⏳ Sauvegarde du modèle fusionné au format Hugging Face...")

merged_dir = "merged_model"
model.save_pretrained(merged_dir)
tokenizer.save_pretrained(merged_dir)

print(f"✅ Modèle fusionné sauvegardé dans '{merged_dir}'")

## --- 5. Cloner et compiler llama.cpp ---

In [ ]:
print("\n⏳ Clonage et compilation de llama.cpp (peut prendre 3-5 minutes)...")

!rm -rf llama.cpp
!git clone --depth 1 https://github.com/ggml-org/llama.cpp
!cd llama.cpp && cmake -B build && cmake --build build --config Release -j 4

print("✅ llama.cpp compilé")

## --- 6. Installer les dépendances Python pour le convertisseur ---

In [ ]:
print("\n⏳ Installation des dépendances Python...")
!pip install -q gguf protobuf numpy
print("✅ Dépendances installées")

## --- 7. ÉTAPE 1 : Convertir en f16 ---

In [ ]:
print(f"\n📌 ÉTAPE 1 : Conversion en GGUF f16...")

f16_file = f"{output_gguf_dir}/dental_model_f16.gguf"

!python llama.cpp/convert_hf_to_gguf.py \
    {merged_dir} \
    --outfile {f16_file} \
    --outtype f16

print(f"✅ Conversion f16 terminée : {f16_file}")

## --- 8. ÉTAPE 2 : Quantifier en q4_k_m ---

In [ ]:
print(f"\n📌 ÉTAPE 2 : Quantification en {quantization_method}...")

output_file = f"{output_gguf_dir}/dental_model_{quantization_method}.gguf"

!./llama.cpp/build/bin/llama-quantize \
    {f16_file} \
    {output_file} \
    {quantization_method}

print(f"✅ Quantification terminée : {output_file}")

## --- 9. Supprimer le fichier f16 (optionnel - pour libérer de l'espace) ---

In [ ]:
print("\n🧹 Suppression du fichier f16 pour libérer de l'espace...")
os.remove(f16_file)
print("✅ Fichier f16 supprimé")

## --- 10. Vérifier les fichiers créés ---

In [ ]:
print("\n📁 Fichiers créés :")

if os.path.exists(output_gguf_dir) and os.listdir(output_gguf_dir):
    total_size = 0
    for file in os.listdir(output_gguf_dir):
        filepath = os.path.join(output_gguf_dir, file)
        size = os.path.getsize(filepath) / (1024 * 1024)
        total_size += size
        print(f"   - {file} ({size:.1f} MB)")
    print(f"\n📊 Taille totale : {total_size:.1f} MB")
else:
    print(f"   ⚠️ Le dossier '{output_gguf_dir}' est vide.")

## --- 11. Créer le Modelfile pour Ollama ---

In [ ]:
print("\n⏳ Création du Modelfile pour Ollama...")

gguf_files = []
if os.path.exists(output_gguf_dir):
    gguf_files = [f for f in os.listdir(output_gguf_dir) if f.endswith('.gguf')]

if gguf_files:
    gguf_filename = gguf_files[0]
else:
    gguf_filename = f"dental_model_{quantization_method}.gguf"
    print(f"   ⚠️ Aucun fichier GGUF trouvé, utilisation du nom par défaut")

modelfile_content = f'''FROM {output_gguf_dir}/{gguf_filename}

TEMPLATE """{{{{ .Prompt }}}}"""

PARAMETER stop "<|im_end|>"
PARAMETER temperature 0.7
PARAMETER top_p 0.9
'''

with open("Modelfile", "w") as f:
    f.write(modelfile_content)

print("✅ Modelfile créé")